In [1]:
import uproot
import pandas as pd
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

# Pick a FlatCAF file

In [2]:
CAFFilePath = "/pnfs/dune/persistent/physicsgroups/dunendsim/abooth/nd-production/MicroProdN4p1/run-cafmaker/MicroProdN4p1_NDComplex_FHC.caf.full.spineonly/CAF.flat/0002000/0002400/MicroProdN4p1_NDComplex_FHC.caf.full.spineonly.0002459.CAF.flat.root"

with uproot.open(CAFFilePath) as f:
    df_caf = f['cafTree'].arrays(library='ak')

In [22]:
df_caf["rec.mc.nu.E"].show(5)

[[17.9, 47.2, 33.8, 57.9, 3.15, 2.46, ..., 6.26, 2.91, 47.1, 2.96, 2.75, 3.47],
 [63.8, 11.2, 4.99, 3.26, 31.8, 15.9, ..., 14.9, 7.88, 5.91, 2.72, 4.71, 41.2],
 [22.2, 1.62, 10.3, 25.1, 15.3, 3.05, ..., 2.35, 1.68, 5.12, 3.97, 3.79, 2.13],
 ...,
 [33.3, 36.9, 4.85, 2.53, 2.64, 16, ..., 2.66, 1.25, 2.79, 4.07, 1.72, 34.6]]


CAFMaker converts GENIE EventRecord into a CAF True neutrino object ([SRTrueInteraction](https://github.com/DUNE/duneanaobj/blob/v03_14_00/duneanaobj/StandardRecord/SRTrueInteraction.h)) along with many other additinal information.
This means each SRTrueInteraction has a corresponding entry from the GENIE tree, and we save the GENIE tree index into [SRTrueInteraction::genieIdx](https://github.com/DUNE/duneanaobj/blob/v03_14_00/duneanaobj/StandardRecord/SRTrueInteraction.h#L50-L53) variable.
The NuSystTree is also ordered in the same way as the input GENIE Tree, so we can run a merging between two dataframes; NuSystTree with the reweights, and CAFTree with CAF variables

Let's first print `genieIdx`:

In [23]:
df_caf["rec.mc.nu.genieIdx"]

<Array [[0, 1, 2, 3, 4, ..., 90, 91, 92, 93, 94], ...] type='13 * var * int64'>

As you can see, each element is increasing by one, which means each of the GENIE EventRecord is converted into CAF objecet one by one.

# NuSystTree

In [9]:
NuSystTreeFilePath = "../Tutorial_Part1/NuSystTree_Updated.root"

DialColumnName_prefix = "DUNEDAS2026ExampleReweighter_NuSystTutorial"
BranchesFromNuSyst = [
    "Enu_true",
    f"tweak_responses_{DialColumnName_prefix}_DialA",
]

with uproot.open(NuSystTreeFilePath) as f:

    # Note that this time we are using awkward array
    df_nusyst = f["events"].arrays(BranchesFromNuSyst, library="ak")


We now want to "merge" NuSyst's reweight column into the CAF dataframe. There are various ways of doing this, but here we do

1. First reshape NuSystTree to match the CAF; i.e., inject the per-spill structure
1. Then copy NuSystTree columns into CAF dataframe

The per-spill structure can be injected by first counting the number of neutrinos in each spill from the CAF dataframe:

In [25]:
arr_NNus = ak.num(df_caf["rec.mc.nu.genieIdx"])
arr_NNus.show(5)

[95,
 84,
 89,
 ...,
 96]


Then, we "reshape" the nusyst Tree, by slicing the "full" neutrino entries by `arr_NNus`:

In [26]:
df_nusyst_per_spill = ak.unflatten(df_nusyst, arr_NNus)
df_nusyst_per_spill.show(5)

[[{Enu_true: 17.9, ...}, {Enu_true: 47.2, ...}, ..., {Enu_true: 3.47, ...}],
 [{Enu_true: 63.8, ...}, {Enu_true: 11.2, ...}, ..., {Enu_true: 41.2, ...}],
 [{Enu_true: 22.2, ...}, {Enu_true: 1.62, ...}, ..., {Enu_true: 2.13, ...}],
 ...,
 [{Enu_true: 33.3, ...}, {Enu_true: 36.9, ...}, ..., {Enu_true: 34.6, ...}]]


Here, we intentionally also reading neutrino energy from NuSystTree (`Enu_true`), which can be used to check the correct matching to the CAF dataframe

The copying can be done by adding a new field.
Let's first add the neutrino energy and see if the matching is done properly.

In [13]:
df_caf["Enu_true_NuSyst"] = df_nusyst_per_spill["Enu_true"]

In [18]:
print("# CAF Enu")
df_caf[["rec.mc.nu.E"]].show(5)
print("# NuSyst Enu")
df_caf[["Enu_true_NuSyst"]].show(5)

# CAF Enu
[{'rec.mc.nu.E': [17.9, 47.2, 33.8, 57.9, ..., 47.1, 2.96, 2.75, 3.47]},
 {'rec.mc.nu.E': [63.8, 11.2, 4.99, 3.26, ..., 5.91, 2.72, 4.71, 41.2]},
 {'rec.mc.nu.E': [22.2, 1.62, 10.3, 25.1, ..., 5.12, 3.97, 3.79, 2.13]},
 ...,
 {'rec.mc.nu.E': [33.3, 36.9, 4.85, 2.53, ..., 2.79, 4.07, 1.72, 34.6]}]
# NuSyst Enu
[{Enu_true_NuSyst: [17.9, 47.2, 33.8, 57.9, ..., 47.1, 2.96, 2.75, 3.47]},
 {Enu_true_NuSyst: [63.8, 11.2, 4.99, 3.26, ..., 5.91, 2.72, 4.71, 41.2]},
 {Enu_true_NuSyst: [22.2, 1.62, 10.3, 25.1, ..., 5.12, 3.97, 3.79, 2.13]},
 ...,
 {Enu_true_NuSyst: [33.3, 36.9, 4.85, 2.53, ..., 2.79, 4.07, 1.72, 34.6]}]


You should see both field showing the same numbers in same order. 
Next, let's add the reweight column.

In [19]:
df_caf["RW_DialA"] = df_nusyst_per_spill[f"tweak_responses_{DialColumnName_prefix}_DialA"]

In [21]:
df_caf["RW_DialA"].show(5)

[[[0.868, 1, 1.13], [-7.99, 1, 9.99], ..., [0.686, ...], [0.973, 1, 1.03]],
 [[0.0352, 1, 1.96], [0.421, 1, 1.58], ..., [0.775, ...], [-0.71, 1, 2.71]],
 [[-4.74, 1, 6.74], [0.294, 1, 1.71], ..., [0.46, ..., 1.54], [0.911, 1, 1.09]],
 ...,
 [[-1.14, 1, 3.14], [-6.13, 1, 8.13], ..., [0.627, ...], [-0.353, 1, 2.35]]]


# Exercise 2-1

Now you have a CAF dataframe with the reweights added from NuSystTree.
Re-do the morning session, CAF selection, but now also draw a reweighted distribution.
How does a reweight in Q2 affects the neutrino energy distribution?